In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error
from scipy.special import j0
from src.custom_fastkan import FastKAN
import pandas as pd

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

Using device: mps


In [2]:
def true_function(x):
    return torch.from_numpy(j0(20 * x.cpu().numpy())).to(x.device).float()

torch.manual_seed(42)
num_samples = 10000

X = torch.rand(num_samples, 1) * 2 - 1 
y = true_function(X)

train_size = int(0.7 * num_samples)
val_size = int(0.15 * num_samples)
test_size = num_samples - train_size - val_size

X_train, X_val, X_test = torch.split(X, [train_size, val_size, test_size])
y_train, y_val, y_test = torch.split(y, [train_size, val_size, test_size])

X_train = X_train.to(device)
y_train = y_train.to(device)
X_val = X_val.to(device)
y_val = y_val.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}, Test shape: {X_test.shape}")

Train shape: torch.Size([7000, 1]), Val shape: torch.Size([1500, 1]), Test shape: torch.Size([1500, 1])


In [3]:
model = FastKAN([1, 1], grid_min=-1.2, grid_max=1.2, num_grids=20, use_base_update=False, use_layernorm=False).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [4]:
print("Training KAN...")
train_losses = []
val_losses = []

for epoch in range(1000):
    optimizer.zero_grad()
    pred = model(X_train)
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    optimizer.step()
    
    if epoch % 50 == 0:
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = torch.mean((val_pred - y_val)**2)
            train_losses.append(loss.item())
            val_losses.append(val_loss.item())
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

model.eval()
with torch.no_grad():
    test_pred = model(X_test)
    mse = mean_squared_error(y_test.cpu(), test_pred.cpu())
    r2 = r2_score(y_test.cpu(), test_pred.cpu())
    
    print(f"KAN Test MSE: {mse:.6f}")
    print(f"KAN Test R2: {r2:.4f}")

Training KAN...
Epoch 0, Train MSE: 0.085805, Val MSE: 0.082497
Epoch 50, Train MSE: 0.015893, Val MSE: 0.015001
Epoch 100, Train MSE: 0.003619, Val MSE: 0.003336
Epoch 150, Train MSE: 0.000958, Val MSE: 0.000868
Epoch 200, Train MSE: 0.000578, Val MSE: 0.000545
Epoch 250, Train MSE: 0.000539, Val MSE: 0.000520
Epoch 300, Train MSE: 0.000532, Val MSE: 0.000517
Epoch 350, Train MSE: 0.000530, Val MSE: 0.000516
Epoch 400, Train MSE: 0.000529, Val MSE: 0.000515
Epoch 450, Train MSE: 0.000529, Val MSE: 0.000515
Epoch 500, Train MSE: 0.000528, Val MSE: 0.000515
Epoch 550, Train MSE: 0.000528, Val MSE: 0.000515
Epoch 600, Train MSE: 0.000527, Val MSE: 0.000514
Epoch 650, Train MSE: 0.000527, Val MSE: 0.000514
Epoch 700, Train MSE: 0.000527, Val MSE: 0.000514
Epoch 750, Train MSE: 0.000526, Val MSE: 0.000513
Epoch 800, Train MSE: 0.000526, Val MSE: 0.000513
Epoch 850, Train MSE: 0.000525, Val MSE: 0.000512
Epoch 900, Train MSE: 0.000525, Val MSE: 0.000512
Epoch 950, Train MSE: 0.000524, Val M

In [5]:
print("\nAnalyzing individual KAN prediction losses...")
individual_losses = []
predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        individual_losses.append(loss)
        predictions.append(prediction)

individual_losses = np.array(individual_losses)
predictions = torch.stack(predictions)
sorted_indices = np.argsort(individual_losses)

mean_loss = np.mean(individual_losses)
lowest_indices = sorted_indices[:3] 
highest_indices = sorted_indices[-3:]

mean_distances = np.abs(individual_losses - mean_loss)
mean_sorted_indices = np.argsort(mean_distances)
mean_indices = mean_sorted_indices[:3]

print(f"\nKAN Loss Statistics:")
print(f"Mean Loss: {mean_loss:.6f}")
print(f"Min Loss: {individual_losses[lowest_indices[0]]:.6f}")
print(f"Max Loss: {individual_losses[highest_indices[-1]]:.6f}")


Analyzing individual KAN prediction losses...

KAN Loss Statistics:
Mean Loss: 0.000531
Min Loss: 0.000000
Max Loss: 0.001953


In [6]:
import torch.nn as nn
torch.manual_seed(42)

class MLP(nn.Module):
    def __init__(self, input_dim=1, hidden_dims=[64, 64], output_dim=1):
        super(MLP, self).__init__()
        layers = []
        curr_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(curr_dim, h_dim))
            layers.append(nn.ReLU())
            curr_dim = h_dim
        layers.append(nn.Linear(curr_dim, output_dim))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

mlp_model = MLP(input_dim=1, hidden_dims=[64, 32], output_dim=1).to(device)
mlp_optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.01)

print("Training MLP...")
for epoch in range(1300):
    mlp_model.train()
    mlp_optimizer.zero_grad()
    pred = mlp_model(X_train)
    loss = torch.mean((pred - y_train)**2)
    loss.backward()
    mlp_optimizer.step()
    
    if epoch % 100 == 0:
        mlp_model.eval()
        with torch.no_grad():
            val_pred = mlp_model(X_val)
            val_loss = torch.mean((val_pred - y_val)**2)
            print(f"Epoch {epoch}, Train MSE: {loss.item():.6f}, Val MSE: {val_loss.item():.6f}")

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(X_test)
    mlp_mse = mean_squared_error(y_test.cpu(), mlp_pred.cpu())
    mlp_r2 = r2_score(y_test.cpu(), mlp_pred.cpu())
    
    print(f"\nMLP Test MSE: {mlp_mse:.6f}")
    print(f"MLP Test R2: {mlp_r2:.4f}")

Training MLP...
Epoch 0, Train MSE: 0.103705, Val MSE: 0.091751
Epoch 100, Train MSE: 0.036273, Val MSE: 0.035520
Epoch 200, Train MSE: 0.017704, Val MSE: 0.017491
Epoch 300, Train MSE: 0.014783, Val MSE: 0.014566
Epoch 400, Train MSE: 0.014470, Val MSE: 0.014263
Epoch 500, Train MSE: 0.013781, Val MSE: 0.013569
Epoch 600, Train MSE: 0.012835, Val MSE: 0.012667
Epoch 700, Train MSE: 0.011476, Val MSE: 0.011329
Epoch 800, Train MSE: 0.006941, Val MSE: 0.006825
Epoch 900, Train MSE: 0.003700, Val MSE: 0.003722
Epoch 1000, Train MSE: 0.002103, Val MSE: 0.002606
Epoch 1100, Train MSE: 0.004414, Val MSE: 0.001805
Epoch 1200, Train MSE: 0.000701, Val MSE: 0.000664

MLP Test MSE: 0.000577
MLP Test R2: 0.9931


In [7]:
print("\nAnalyzing individual MLP prediction losses...")
mlp_individual_losses = []
mlp_predictions = []
with torch.no_grad():
    for i in range(len(X_test)):
        input_seq = X_test[i]
        ground_truth = y_test[i]
        prediction = mlp_model(input_seq.unsqueeze(0)).squeeze()
        
        loss = ((prediction - ground_truth) ** 2).item()
        mlp_individual_losses.append(loss)
        mlp_predictions.append(prediction)

mlp_individual_losses = np.array(mlp_individual_losses)
mlp_predictions = torch.stack(mlp_predictions)
mlp_sorted_indices = np.argsort(mlp_individual_losses)

mlp_mean_loss = np.mean(mlp_individual_losses)
mlp_lowest_indices = mlp_sorted_indices[:3] 
mlp_highest_indices = mlp_sorted_indices[-3:]

mlp_mean_distances = np.abs(mlp_individual_losses - mlp_mean_loss)
mlp_mean_sorted_indices = np.argsort(mlp_mean_distances)
mlp_mean_indices = mlp_mean_sorted_indices[:3]

print(f"\nMLP Loss Statistics:")
print(f"Mean Loss: {mlp_mean_loss:.6f}")
print(f"Min Loss: {mlp_individual_losses[mlp_lowest_indices[0]]:.6f}")
print(f"Max Loss: {mlp_individual_losses[mlp_highest_indices[-1]]:.6f}")


Analyzing individual MLP prediction losses...



MLP Loss Statistics:
Mean Loss: 0.000577
Min Loss: 0.000000
Max Loss: 0.006979


In [8]:
table_data = []
categories = [("Lowest", lowest_indices), ("Highest", highest_indices), ("Mean", mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data.append({
            "Category": label,
            "Index": idx,
            "Input (x)": f"({X_test[idx][0].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": predictions[idx].item(),
            "Loss": individual_losses[idx]
        })

df_kan_analysis = pd.DataFrame(table_data)
df_kan_analysis

,Category,Index,Input (x),True Value,Predicted,Loss
0,Lowest,722,(0.9648),0.170925,0.170909,2.369414e-10
1,Lowest,415,(0.9649),0.170947,0.170924,5.578261e-10
2,Lowest,1169,(0.5668),-0.104387,-0.104416,8.211052e-10
3,Highest,880,(-0.0023),0.999486,0.955416,1.942218e-03
4,Highest,796,(-0.0019),0.999636,0.955532,1.945130e-03
5,Highest,970,(0.0000),1.000000,0.955810,1.952755e-03
6,Mean,337,(0.3759),0.263970,0.240942,5.302840e-04
7,Mean,1037,(0.7464),0.000626,0.023639,5.295993e-04
8,Mean,205,(-0.3313),0.277264,0.300261,5.288699e-04


In [9]:
table_data_mlp = []
categories = [("Lowest", mlp_lowest_indices), ("Highest", mlp_highest_indices), ("Mean", mlp_mean_indices)]

for label, indices in categories:
    for idx in indices:
        table_data_mlp.append({
            "Category": label,
            "Index": idx,
            "Input (x)": f"({X_test[idx][0].item():.4f})",
            "True Value": y_test[idx].item(),
            "Predicted": mlp_predictions[idx].item(),
            "Loss": mlp_individual_losses[idx]
        })

df_mlp_analysis = pd.DataFrame(table_data_mlp)
df_mlp_analysis

,Category,Index,Input (x),True Value,Predicted,Loss
0,Lowest,899,(0.1386),-0.173360,-0.173354,4.010681e-11
1,Lowest,659,(-0.1328),-0.123027,-0.122996,9.541988e-10
2,Lowest,734,(-0.7498),-0.013453,-0.013491,1.462234e-09
3,Highest,194,(0.9998),0.167346,0.246903,6.329371e-03
4,Highest,1113,(-0.9992),0.168035,0.250399,6.783810e-03
5,Highest,1138,(-0.9995),0.167656,0.251198,6.979212e-03
6,Mean,1432,(0.2428),-0.223133,-0.199100,5.775684e-04
7,Mean,1479,(-0.9578),0.161249,0.137234,5.767544e-04
8,Mean,1480,(0.4101),0.121663,0.097656,5.763114e-04


In [10]:
mlp_save_path = "model_pkls/functionbessel_mlp_model.pkl"
torch.save({
    'model_state_dict': mlp_model.state_dict(),
    'config': {
        'input_dim': 1,
        'hidden_dims': [64, 32],
        'output_dim': 1
    }
}, mlp_save_path)
print(f"MLP Model saved to {mlp_save_path}")

MLP Model saved to model_pkls/functionbessel_mlp_model.pkl


In [11]:
kan_save_path = "model_pkls/functionbessel_kan_model.pkl"
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {
        'layers_hidden': [1, 1],
        'grid_min': -1.2,
        'grid_max': 1.2,
        'num_grids': 20,
        'use_base_update': False,
        'use_layernorm': False,
    }
}, kan_save_path)
print(f"KAN Model saved to {kan_save_path}")

KAN Model saved to model_pkls/functionbessel_kan_model.pkl
